# Malware as images — transfer learning and knowledge distillation

The image-based malware-classification side track, in the order it was done: **A** — the generic PyTorch transfer-learning tutorial (VGG16 features on ants-vs-bees) that the rest adapts. **B** — the same recipe on the Malimg dataset: VGG16 features of malware-family images, classified with random forest, logistic regression, SVM and gradient boosting (the second version's different data-loading path reaches the same pipeline and is not repeated). **C** — knowledge distillation on Malimg: a light CNN student trained from a deep teacher with cosine-similarity loss, several loss weights, and sequential variants. **D** — a first look at the Kaggle IoT-malware image dataset.

**Where this fits:** this section is the study material behind the thesis work — the tutorial chapters and side experiments it grew out of. It is coursework, kept for completeness and clearly separate from the pipeline in sections 01–06; the image-domain experiments here are the image counterparts of the feature-space work in `03-attacks/`.

*Consolidated from `TRANSFER_LEARNING_COMPUTER_VISION.ipynb`, `Malware_Classification_transfer_learning.ipynb`, `Knowledge_Distillation.ipynb` and `Kaggle_IOT_Malware.ipynb` (originally in `Adversarials`). Outputs cleared.*

## Part A — The generic tutorial: VGG16 features, ants vs bees

https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

In [ ]:
!wget 'https://download.pytorch.org/tutorial/hymenoptera_data.zip'
!unzip "/content/hymenoptera_data.zip" -d "/content/"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
import seaborn as sns

cudnn.benchmark = True
plt.ion()   # interactive mode

In [ ]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/hymenoptera_data'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated


# Get a batch of training data
inputs, classes = next(iter(dataloaders['train']))

# Make a grid from batch
out = torchvision.utils.make_grid(inputs)

imshow(out, title=[class_names[x] for x in classes])

In [ ]:
from torchvision import models
model = models.vgg16(weights='DEFAULT')

# Freeze model weights
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model

In [ ]:
model.classifier[-1]

In [ ]:
model.classifier = model.classifier[:-1]
model

In [ ]:
features = []
labels = []
model.eval()
with torch.no_grad():
    for inputs, labels_batch in dataloaders['train']:
        inputs = inputs.to(device)
        labels_batch = labels_batch.to(device)
        features_batch = model(inputs)
        features.append(features_batch.cpu().numpy())
        labels.append(labels_batch.cpu().numpy())
        break

In [ ]:
features_batch.shape

In [ ]:
labels_batch

In [ ]:
# Define a function to extract features from the VGG16 model
def extract_vgg16_features(model, dataloader):
    features = []
    labels = []
    model.eval()
    with torch.no_grad():
        for inputs, labels_batch in dataloader:
            inputs = inputs.to(device)
            labels_batch = labels_batch.to(device)
            features_batch = model(inputs)
            features.append(features_batch.cpu().numpy())
            labels.append(labels_batch.cpu().numpy())
    return np.vstack(features), np.hstack(labels)

In [ ]:
# Set device (GPU if available)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Extract features from VGG16
train_features, train_labels = extract_vgg16_features(model, dataloaders['train'])
val_features, val_labels = extract_vgg16_features(model, dataloaders['val'])

In [ ]:
train_features.shape

In [ ]:
val_features.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(train_features, train_labels)

# Evaluate the classifier
val_predictions = clf.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
val_report = classification_report(val_labels, val_predictions)

print("Validation Accuracy:", val_accuracy)
print("Validation Classification Report:\n", val_report)

#Confusion Matrix - verify accuracy of each class
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, val_predictions)
#print(cm)
sns.heatmap(cm, annot=True)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
# Train a Random Forest classifier
clf = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0).fit(train_features, train_labels)


# Evaluate the classifier
val_predictions = clf.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
val_report = classification_report(val_labels, val_predictions)

print("Validation Accuracy:", val_accuracy)
print("Validation Classification Report:\n", val_report)

#Confusion Matrix - verify accuracy of each class
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, val_predictions)
#print(cm)
sns.heatmap(cm, annot=True)

In [ ]:
from sklearn import svm
clf = svm.SVC()
clf.fit(train_features, train_labels)

# Evaluate the classifier
val_predictions = clf.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
val_report = classification_report(val_labels, val_predictions)

print("Validation Accuracy:", val_accuracy)
print("Validation Classification Report:\n", val_report)

#Confusion Matrix - verify accuracy of each class
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, val_predictions)
#print(cm)
sns.heatmap(cm, annot=True)

## Part B — Transfer learning on Malimg (malware-family images)

In [ ]:
!wget "https://www.dropbox.com/s/ep8qjakfwh1rzk4/malimg_dataset.zip"

In [ ]:
!unzip "/content/malimg_dataset.zip" -d "/content/"

In [ ]:
!rm -r '/content/malimg_paper_dataset_imgs'

In [ ]:
!rm -r '/content/dataset'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
import seaborn as sns

In [ ]:
import os
import shutil
import random

# Set the path to your parent dataset directory
parent_dataset_dir = '/content/malimg_paper_dataset_imgs'
main_dir = '/content/dataset/'
# Set the ratio for the train-test split
split_ratio = 0.7  # 70% for training, 30% for testing

# Create train and test directories
train_dir = os.path.join(main_dir, 'train')
test_dir = os.path.join(main_dir, 'val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Iterate through each class folder
for class_folder in os.listdir(parent_dataset_dir):
    class_path = os.path.join(parent_dataset_dir, class_folder)

    # Check if the item is a directory (class folder)
    if os.path.isdir(class_path):
        # Get a list of all images in the class folder
        images = os.listdir(class_path)

        # Shuffle the images randomly
        random.shuffle(images)

        # Determine the split point based on the ratio
        split_point = int(len(images) * split_ratio)

        # Split the images into train and test sets
        train_images = images[:split_point]
        test_images = images[split_point:]

        # Create train and test class folders
        train_class_dir = os.path.join(train_dir, class_folder)
        test_class_dir = os.path.join(test_dir, class_folder)

        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(test_class_dir, exist_ok=True)

        # Move images to the appropriate folders
        for image_file in train_images:
            src = os.path.join(class_path, image_file)
            dst = os.path.join(train_class_dir, image_file)
            shutil.move(src, dst)

        for image_file in test_images:
            src = os.path.join(class_path, image_file)
            dst = os.path.join(test_class_dir, image_file)
            shutil.move(src, dst)

print("Dataset split into train and test sets successfully.")

In [ ]:
from PIL import Image

image = Image.open("/content/dataset/train/Adialer.C/000e30a0819ac7ed931d629ce2ef8948.png")
pixel_data = image.getdata()

# Pixel data is a list of tuples, where each tuple represents a pixel.
# Each tuple contains four values: red, green, blue, and alpha.

print(pixel_data[0])

In [ ]:
image

In [ ]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/dataset'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=10000,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#IMPORTANT : our data has just one channel , so other two channels are created based on a copy of the first channel

In [ ]:
dataset_sizes

In [ ]:
# Get a batch of training data
inputs, classes = next(iter(dataloaders['train']))
print(inputs.shape)
print(classes.shape)

In [ ]:
image_datasets['train'].class_to_idx

In [ ]:
from torchvision import models
model = models.vgg16(weights='DEFAULT')

# Freeze model weights
for param in model.parameters():
    param.requires_grad = False

model.classifier = model.classifier[:-1]
model

In [ ]:
print(train_features.shape)
print(val_features.shape)

In [ ]:
# number of zeros in features
for i in range(10):
  sample = train_features[i]
  print('label : ',train_labels[i])
  print('number of zero nerons : ',sum(sample==0))
  print('---------------------------------')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search over
params_to_test = {
    'n_estimators': [100, 300, 600, 1000], # The number of trees in the forest
    #'max_depth': [ 1, 5, 10, 25], # The maximum depth of the tree
    #'max_features': np.arange(0.1, 1.1, 0.1) # The number of features to consider at each split
}

# Create a random forest classifier object
rf_model = RandomForestClassifier(random_state=42)

# Create a grid search object with 10-fold cross-validation
grid_search = GridSearchCV(rf_model, param_grid=params_to_test, cv=10, scoring='accuracy', n_jobs=-1)

# Fit the grid search on the training data
grid_search.fit(train_features, train_labels)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Print the results
print(f'Best parameters: {best_params}')
print(f'Best score: {best_score}')

In [ ]:
# Train the final model with the best hyperparameters
best_rf = RandomForestClassifier(**best_params)
best_rf.fit(train_features, train_labels)

# Evaluate the final model on the test set
val_predictions = best_rf.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
val_report = classification_report(val_labels, val_predictions)

print("Validation Accuracy:", val_accuracy)
print("Validation Classification Report:\n", val_report)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
clf = LogisticRegression(random_state=0).fit(train_features, train_labels)

# Evaluate the classifier
val_predictions = clf.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
val_report = classification_report(val_labels, val_predictions)

print("Validation Accuracy:", val_accuracy)
print("Validation Classification Report:\n", val_report)

#Confusion Matrix - verify accuracy of each class
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, val_predictions)
#print(cm)
sns.heatmap(cm, annot=True)

## Part C — Knowledge distillation on Malimg

# light model

In [ ]:
class lightCNN(nn.Module):
    def __init__(self, num_classes=25):
        super(lightCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate the  model
model = lightCNN()
model = model.to(device)
# Print the model architecture
print(model)

In [ ]:
total_params_deep = "{:,}".format(sum(p.numel() for p in model.parameters()))
print(f"model parameters: {total_params_deep}")

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
batch_size=64
for epoch in range(25):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(dataloaders['train'], 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs[:,0:1]) # just first channel
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    print(f'{epoch + 1} loss: {(running_loss *1000) / (len(dataloaders["train"]) * batch_size):.3f}')



print('Finished Training')

In [ ]:
PATH = 'lightCNN.pth'
torch.save(model.state_dict(), PATH)

#model = lightCNN()
#model.load_state_dict(torch.load(PATH))

In [ ]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in dataloaders['val']:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        # calculate outputs by running images through the network
        outputs = model(images[:,0:1])
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on test images: {100 * correct / total} %')

In [ ]:
# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in class_names}
total_pred = {classname: 0 for classname in class_names}

# again no gradients needed
with torch.no_grad():
    for data in dataloaders['val']:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images[:,0:1])
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[class_names[label]] += 1
            total_pred[class_names[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

# Knowledge_Distillation

https://pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html#prerequisites

In [ ]:
class ModifiedLightNNCosine(nn.Module):
    def __init__(self, num_classes=25):
        super(ModifiedLightNNCosine, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        return x, flattened_conv_output

# Instantiate the  model
torch.manual_seed(42)
modified_nn_light  = ModifiedLightNNCosine().to(device)
# Print the model architecture
print(modified_nn_light)

In [ ]:
from torchvision import models
modified_nn_deep  = models.vgg16(weights='DEFAULT')

# Freeze model weights
for param in modified_nn_deep.parameters():
    param.requires_grad = False

modified_nn_deep.classifier = modified_nn_deep.classifier[:-1]
modified_nn_deep = modified_nn_deep.to(device)
modified_nn_deep

In [ ]:
total_params_deep = "{:,}".format(sum(p.numel() for p in modified_nn_deep.parameters()))
print(f"model parameters: {total_params_deep}")

In [ ]:
def train_cosine_loss(teacher, student, train_loader, epochs, learning_rate, hidden_rep_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs[:,0:1])

            # Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is the case where loss minimization leads to cosine similarity increase.
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device))

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [ ]:
train_cosine_loss(teacher=modified_nn_deep, student=modified_nn_light, train_loader=dataloaders['train'], epochs=25, learning_rate=0.001, hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)

In [ ]:
def test_multiple_outputs(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs, _ = model(inputs[:,0:1]) # Disregard the second tensor of the tuple
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [ ]:
test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, dataloaders['val'], device)

In [ ]:
# with different loss_weight
# Instantiate the  model
torch.manual_seed(42)
modified_nn_light  = ModifiedLightNNCosine().to(device)
train_cosine_loss(teacher=modified_nn_deep, student=modified_nn_light, train_loader=dataloaders['train'], epochs=25, learning_rate=0.001, hidden_rep_loss_weight=0.5, ce_loss_weight=0.5, device=device)
test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, dataloaders['val'], device)

In [ ]:
def train_cosine_loss_sequence(teacher, student, train_loader, epochs, learning_rate, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs[:,0:1])

            # Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is the case where loss minimization leads to cosine similarity increase.
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device))

            # Weighted sum of the two losses
            loss = hidden_rep_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


    for epoch in range(epochs):
        student.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs[:,0:1])

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")
        test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, dataloaders['val'], device)
        print('------------------------------------------------------------------------------------')

In [ ]:
# train first by knwledge distillation , then train the light model by itself
# Instantiate the  model
torch.manual_seed(42)
modified_nn_light  = ModifiedLightNNCosine().to(device)
train_cosine_loss_sequence(teacher=modified_nn_deep, student=modified_nn_light, train_loader=dataloaders['train'], epochs=25, learning_rate=0.001, device=device)

In [ ]:
def train_cosine_loss_sequence2(teacher, student, train_loader, epochs, learning_rate, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    #train the feature part(first part)
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs[:,0:1])

            # Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is the case where loss minimization leads to cosine similarity increase.
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device))

            # Weighted sum of the two losses
            loss = hidden_rep_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


    #train the clssifier part

    # Freeze the feature layers by setting requires_grad to False
    for param in student.features.parameters():
        param.requires_grad = False

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs[:,0:1])

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

    #train all the model

    # UNFreeze the feature layers by setting requires_grad to False
    for param in student.features.parameters():
        param.requires_grad = True

    for epoch in range(epochs):
            student.train()
            running_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()

                # Forward pass with the teacher model and keep only the hidden representation
                with torch.no_grad():
                    teacher_hidden_representation = teacher(inputs)

                # Forward pass with the student model
                student_logits, student_hidden_representation = student(inputs[:,0:1])

                # Calculate the true label loss
                label_loss = ce_loss(student_logits, labels)

                # Weighted sum of the two losses
                loss = label_loss

                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")
            test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, dataloaders['val'], device)
            print('------------------------------------------------------------------------------------')

In [ ]:
# train first by knwledge distillation , then train the light model by itself
# Instantiate the  model
torch.manual_seed(42)
modified_nn_light  = ModifiedLightNNCosine().to(device)
train_cosine_loss_sequence2(teacher=modified_nn_deep, student=modified_nn_light, train_loader=dataloaders['train'], epochs=25, learning_rate=0.001, device=device)

## Part D — The Kaggle IoT-malware images: first look

In [ ]:
# Install Kaggle API.
!pip install kaggle
# Run the following code to configure the path to “kaggle.json”
import os
#!chmod 600 /content/kaggle.json
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
!kaggle datasets download -d anaselmasry/iot-malware/

In [ ]:
!unzip iot-malware.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
import seaborn as sns

import os
import shutil
import random

In [ ]:
parent_dataset_dir = '/content/IOT_Malware_dataset'
os.listdir(parent_dataset_dir)

In [ ]:
# Set the path to your parent dataset directory
parent_dataset_dir = '/content/IOT_Malware_dataset'
main_dir = '/content/dataset/'
# Set the ratio for the train-test split
split_ratio = 0.8  # 70% for training, 30% for testing

# Create train and test directories
train_dir = os.path.join(main_dir, 'train')
test_dir = os.path.join(main_dir, 'val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Iterate through each class folder
for class_folder in os.listdir(parent_dataset_dir):
    class_path = os.path.join(parent_dataset_dir, class_folder)

    # Check if the item is a directory (class folder)
    if os.path.isdir(class_path):
        # Get a list of all images in the class folder
        images = os.listdir(class_path)

        # Shuffle the images randomly
        random.shuffle(images)

        # Determine the split point based on the ratio
        split_point = int(len(images) * split_ratio)

        # Split the images into train and test sets
        train_images = images[:split_point]
        test_images = images[split_point:]

        # Create train and test class folders
        train_class_dir = os.path.join(train_dir, class_folder)
        test_class_dir = os.path.join(test_dir, class_folder)

        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(test_class_dir, exist_ok=True)

        # Move images to the appropriate folders
        for image_file in train_images:
            src = os.path.join(class_path, image_file)
            dst = os.path.join(train_class_dir, image_file)
            shutil.move(src, dst)

        for image_file in test_images:
            src = os.path.join(class_path, image_file)
            dst = os.path.join(test_class_dir, image_file)
            shutil.move(src, dst)

print("Dataset split into train and test sets successfully.")

In [ ]:
from PIL import Image

image = Image.open("/content/dataset/train/Benign/009246275d77da1dd0c9cbcf19a5b8e3_0.jpg")

pixel_data = image.getdata()
# Pixel data is a list of tuples, where each tuple represents a pixel.
# Each tuple contains four values: red, green, blue, and alpha.
print(len(pixel_data))
print(pixel_data[0])

In [ ]:
image_tensor = torchvision.transforms.ToTensor()(image)
print(image_tensor.shape)

In [ ]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/dataset'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=128,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(class_names)
print(dataset_sizes)